In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_image(title, img, cmap='gray'):
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

img = cv2.imread('moon.png', cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError("Could not load moon.png")

img = img.astype(np.float32) / 255.0
M, N = img.shape

show_image("Image", img)

P = cv2.getOptimalDFTSize(2 * M)
Q = cv2.getOptimalDFTSize(2 * N)

padded = np.zeros((P, Q), dtype=np.float32)
padded[:M, :N] = img

show_image("Pad", padded)

u = np.fft.fftfreq(P).reshape(P, 1)
v = np.fft.fftfreq(Q).reshape(1, Q)

U, V = np.meshgrid(u, v, indexing='ij')
D2 = U**2 + V**2

laplacian_filter = -4 * np.pi**2 * D2

show_image(
    "Laplacian Filter",
    np.fft.fftshift(np.abs(laplacian_filter))
)

F = np.fft.fft2(padded)
F_lap = F * laplacian_filter

g_p = np.real(np.fft.ifft2(F_lap))

show_image("Laplacian", g_p[:M, :N])

g_cropped = g_p[:M, :N]

alpha = 0.5
sharpened = img + alpha * g_cropped
sharpened = np.clip(sharpened, 0, 1)

show_image("Image + Laplacian", sharpened)